In [ ]:
# !pip3 install vitaldb

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import vitaldb as v

In [ ]:
# from TTransformer import (
#     EmbeddingLayer, 
#     PositionalEncoding, 
#     TransformerBlock, 
#     RegressionHead, 
#     ForecastingModel
# )

# Data overview

In [ ]:
track_names = ["SNUADC/ART",        # Arterial pressure wave  | W/500 | mmHg
               "SNUADC/ECG_II",     # ECG lead II wave        | W/500 | mV
               "SNUADC/ECG_V5",     # ECG lead V5 wave        | W/500 | mV
               "BIS/EEG1_WAV",      # EEG wave from channel 1 | W/128 | uV
               "BIS/EEG2_WAV",      # EEG wave from channel 2 | W/128 | uV
               "Solar8000/RR_CO2",  # Respiratory rate based on capnography | N | /min
               "Primus/CO2",        # Capnography wave        | W/62.5 | mmHg
               "BIS/BIS",           # Bispectral index value  |    N   | unitless
               ]

# read 1 patient from the VitalDB server
patient = v.VitalFile(1, track_names)  # should we add max length?
# convert to pandas dataframe
patient = patient.to_pandas(track_names=track_names, interval=5)
# change column names to 'human-readable'
patient.columns = ['arterial_pres', 'ecg1', 'ecg2', 'eeg1', 'eeg2', 'resp_rate', 'capnography', 'bis']

In [ ]:
patient

In [ ]:
plt.plot(patient.bis)
plt.title('BIS [N]')
plt.xlabel('sample')
plt.ylabel('BIS value')
plt.grid()

In [ ]:
plt.plot(patient.arterial_pres)
plt.title('Arterial pressure wave')
plt.xlabel('Sample')
plt.ylabel('Arterial pressure [mmHg]')

In [ ]:
# sharex=True ensures both plots zoom/scroll together
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

# 2. Plot the first EEG channel
ax1.plot(patient.ecg1, color='royalblue', linewidth=0.8)
ax1.set_ylabel('ECG 1 [mV]')
ax1.set_title('ECG Waveforms', fontsize=14)
ax1.grid(True, alpha=0.3)

# 3. Plot the second EEG channel
ax2.plot(patient.ecg2, color='crimson', linewidth=0.8)
ax2.set_ylabel('ECG 2 [mV]')
ax2.set_xlabel('Sample Index')
ax2.grid(True, alpha=0.3)

# 4. Remove white space between plots and show
plt.tight_layout()
plt.show()

In [ ]:
# 1. Create the figure and subplots
# sharex=True ensures both plots zoom/scroll together
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

# 2. Plot the first EEG channel
ax1.plot(patient.eeg1, color='royalblue', linewidth=0.8)
ax1.set_ylabel('EEG 1 [μV]')
ax1.set_title('EEG Waveforms', fontsize=14)
ax1.grid(True, alpha=0.3)

# 3. Plot the second EEG channel
ax2.plot(patient.eeg2, color='crimson', linewidth=0.8)
ax2.set_ylabel('EEG 2 [μV]')
ax2.set_xlabel('Sample Index')
ax2.grid(True, alpha=0.3)

# 4. Remove white space between plots and show
plt.tight_layout()
plt.show()

In [ ]:
plt.plot(patient.resp_rate) # is this useless ????
plt.title('Respiratory rate (N/min)')
plt.xlabel('Sample')
plt.ylabel('Respiratory rate')

In [ ]:
plt.plot(patient.capnography) # is this useless ????
plt.title('Capnography wave')
plt.xlabel('Sample')
plt.ylabel('Capnography wave [mmHg]')

In [ ]:
import matplotlib.pyplot as plt

cols_to_plot = [
    'arterial_pres', 
    'ecg1', 
    'ecg2', 
    'eeg1', 
    'eeg2', 
    'capnography', 
    'resp_rate', 
    'bis'
]

fig, axs = plt.subplots(len(cols_to_plot), 1, figsize=(15, 18), sharex=True)

for i, col in enumerate(cols_to_plot):
    data = patient[col].copy()
        
    axs[i].plot(data, linewidth=0.8)
    axs[i].set_ylabel(col, fontsize=10)
    axs[i].grid(True, alpha=0.3)

axs[-1].set_xlabel('Sample Index', fontsize=12)
plt.suptitle('Patient Multi-Parameter Monitoring', fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.98])

plt.show()

# Transformer training 
Preprocess only BIS index - filter values > 0

In [ ]:
patient_bis_nonzero = patient[patient['bis'] > 0] # also filters out NaNs

print('Original size:', len(patient.bis), '\nSize after filtering "bis > 0":', len(patient_bis_nonzero.bis))
print(len(patient_bis_nonzero))
plt.plot(patient_bis_nonzero.bis)
plt.title('BIS [N]')
plt.xlabel('sample')
plt.ylabel('BIS value')
plt.grid()

In [ ]:
NUMBER_OF_PATIENTS = 11
patients = []
for i in range(1, NUMBER_OF_PATIENTS):
    p = v.VitalFile(i, track_names)
    p = p.to_pandas(track_names=track_names, interval=5)
    p.columns = ['arterial_pres', 'ecg1', 'ecg2', 'eeg1', 'eeg2', 'resp_rate', 'capnography', 'bis']
    p = p[p['bis'] > 0]
    patients.append(p)

In [ ]:
for i, p in enumerate(patients):
    print(i, len(p))
    # print(p.head(1))

### Normalize with Z-score

In [ ]:
seq_len = 200
x1 = patients[0].eeg1
x1_mean = np.mean(x1)
x1_std = np.std(x1)
x1_z = [(x1_i - x1_mean)/x1_std for x1_i in x1]

x2 = patients[0].eeg2
x2_mean = np.mean(x2)
x2_std = np.std(x2)
x2_z = [(x2_i - x2_mean)/x2_std for x2_i in x2]

x_z = np.stack([x1_z, x2_z], axis=1) # [[x1[0],x2[0]], [x1[1], x2[1]]...]

In [ ]:
print(x1_z[0], x2_z[0])
print(x_z[0])

In [ ]:


X = np.array([x_z[i:i+seq_len] for i in range(x_z.shape[0]-seq_len)]).reshape(-1, seq_len, 2)

y = patients[0].bis
y_mean = np.mean(y)
y_std = np.std(y)
y_z = [(y_i - y_mean)/y_std for y_i in y]
Y = np.array([y_z[i+seq_len] for i in range(len(y_z)-seq_len)]).reshape(-1, 1)

print(f"x_z: mean {np.mean(x_z)} std {np.std(x_z)}  y_z: mean {np.mean(y_z)} std{np.std(y_z)}")
print(f"X: {X.shape}, Y: {Y.shape}")



In [ ]:
patients[0].head(5)

### Forecasting model

In [ ]:
class EmbeddingLayer(nn.Module):
    def __init__(self, input_size=1, embed_size=8, conv1d_emb=False, kernel_size=3):
        super().__init__()
        self.input_size = input_size
        self.embed_size = embed_size
        self.conv1d_emb = conv1d_emb
        self.kernel_size = kernel_size
        self.padding = kernel_size - 1

        if conv1d_emb:
            self.conv = nn.Conv1d(in_channels=input_size, out_channels=embed_size, kernel_size=kernel_size)
        else:
            self.input_embedding = nn.Linear(input_size, embed_size)
    
    def forward(self, x):
        # shape: (batch, 200, 1)
        if self.conv1d_emb:
            # pad the START of the sequence (dimension 1) with self.padding zeros
            # F.pad(tensor, (left, right, top, bottom), value) — but for 3D tensors
            # the pad tuple goes from last dim backwards: (dim2_left, dim2_right, dim1_left, dim1_right)
            # torch.nn.functional.pad(input, pad, mode='constant', value=None)
            x = F.pad(x, (0, 0, self.padding, 0), "constant", 0)
            # Conv1d expects (batch, channels, length) — transpose dims 1 and 2
            x = x.transpose(2,1)
            # apply convolution
            x = self.conv(x)
            # transpose back to (batch, seq_len, embed_size)
            x = x.transpose(2,1)
        else:
            x = self.input_embedding(x)
        
        return x

In [ ]:
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, 1, d_model)
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0)/d_model))

        pe[:, 0, 0::2] = torch.sin(position * div_term)
        pe[:, 0, 1::2] = torch.cos(position * div_term)

        pe = pe.transpose(0,1)
        self.register_buffer('pe',pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

In [ ]:
from torch.nn.modules.transformer import TransformerEncoderLayer


class TransformerBlock(nn.Module):
    def __init__(self, embed_size, nhead, dim_feedforward, dropout, seq_len, device):
        super().__init__()
        self.seq_len = seq_len
        self.device = device

        self.encoder_layer = TransformerEncoderLayer(
            d_model=embed_size,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )

    def forward(self, x):
        mask = torch.triu(
            torch.full((x.size(1),x.size(1)), float('-inf'), device=self.device),
            diagonal = 1
        )

        x = self.encoder_layer(x, src_mask=mask)
        return x

In [ ]:
class RegressionHead(nn.Module):
    def __init__(self, seq_len, embed_size, dim_feedforward, dropout):
        super().__init__()

        flat_size = seq_len * embed_size

        self.linear1 = nn.Linear(flat_size, dim_feedforward)
        self.linear2 = nn.Linear(int(dim_feedforward), int(dim_feedforward/2))
        self.linear3 = nn.Linear(int(dim_feedforward/2), int(dim_feedforward/4))
        self.linear4 = nn.Linear(int(dim_feedforward/4), int(dim_feedforward/16))
        self.linear5 = nn.Linear(int(dim_feedforward/16), int(dim_feedforward/64))
        self.outlayer = nn.Linear(int(dim_feedforward/64), 1)

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # shape (batch, seq_len, embed_size)

        # flatten everything except batch dim
        x = x.reshape(x.size(0), x.size(1)*x.size(2))

        # pass through each linear layer with relu + dropout
        x = self.linear1(x)
        x = self.relu(x)
        x = self.dropout(x)

        x = self.linear2(x)
        x = self.relu(x)
        x = self.dropout(x)

        x = self.linear3(x)
        x = self.relu(x)
        x = self.dropout(x)

        x = self.linear4(x)
        x = self.relu(x)
        x = self.dropout(x)

        x = self.linear5(x)
        x = self.relu(x)
        x = self.dropout(x)

        return self.outlayer(x)


In [ ]:

class ForecastingModel(nn.Module):
    def __init__(self, 
                input_size,
                seq_len, 
                embed_size, 
                nhead, 
                dim_feedforward, 
                dropout, 
                conv1d_emb=True,
                kernel_size=3,
                device='cpu'):
        super().__init__()
        
        self.input_size = input_size
        self.seq_len = seq_len
        self.embed_size = embed_size
        self.nhead = nhead
        self.dim_feedforward = dim_feedforward
        self.dropout = dropout
        self.conv1d_emb = conv1d_emb
        self.kernel_size = kernel_size
        self.device = device

        self.embedding_layer = EmbeddingLayer(
            input_size=input_size,
            embed_size=embed_size,
            conv1d_emb=conv1d_emb,
            kernel_size=kernel_size
        )

        self.pos_encoding = PositionalEncoding(embed_size, dropout)

        self.transformer_block = TransformerBlock(
            embed_size=embed_size, 
            nhead=nhead, 
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            seq_len=seq_len,
            device=device
        )

        self.regression_head = RegressionHead(
            seq_len=seq_len,
            embed_size=embed_size,
            dim_feedforward=dim_feedforward,
            dropout=dropout
        )


    def forward(self, x):
        x = self.embedding_layer(x)
        x = self.pos_encoding(x)
        x = self.transformer_block(x)
        x = self.regression_head(x)
        return x

### Training

In [ ]:
from torch.optim.lr_scheduler import ExponentialLR

EPOCHS = 30
BATCH_SIZE = 1
LEARNING_RATE = 2.2e-6
device = 'cuda'

model = ForecastingModel(input_size=2, seq_len=200, embed_size=128, nhead=32,
                         dim_feedforward=1024, dropout=0, device=device)

model.to(device)
model.train()

criterion = torch.nn.HuberLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
scheduler = ExponentialLR(optimizer, gamma=0.9)

dataset = TensorDataset(torch.tensor(X, dtype=torch.float32).to(device), 
                        torch.tensor(Y, dtype=torch.float32).to(device))
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE)

for epoch in range(EPOCHS):
    for batch_x, batch_y in dataloader:
        optimizer.zero_grad()
        y_hat = model(batch_x)
        loss = criterion(y_hat, batch_y)
        loss.backward()
        optimizer.step()

    scheduler.step()
    print(f"Epoch {epoch+1}/{EPOCHS}: Loss={loss.item():.6f}")
